# Path B — C+D notebook (10 epochs each, oracle aux loss enabled)

Runs the two oracle-pressure arms. Pair with `train_path_b_colab_AB.ipynb` (running in parallel on a second Colab) to cover all four runs in ~24h.

| run | --oracle-lambda | --color-augment | purpose |
|---|---|---|---|
| **C** | 0.05 | yes | Conservative oracle correction |
| **D** | 0.10 | yes | Aggressive oracle correction |

Compares against runs A/B in the parallel notebook. Final ablation logic:
- If D > C > B > A → oracle is real and dose-responsive
- If C ≈ D → λ=0.05 is enough
- If B ≫ A → gain was color symmetry, not oracle

**Watch the oracle val metrics each epoch.** `top1≥.15` and `KL_weighted` should move IF oracle is doing real work. If they don't move by epoch 5, λ is too small. If V12 val_loss diverges from B's trajectory, λ is too large.

**Batch size:** 32768 (same as pillar2z). If OOM, drop to 16384 + halve `--lr`. The oracle batch (4096) is added on top of the V12 batch via concat, so peak memory is ~12% higher than A/B — adjust if needed.

## Upload to Drive (`MyDrive/alphatrain/`)

1. `colorlines_path_b.tar.gz` — code + oracle tensor (~1.1 MB)
2. `v12_pillar2z.pt.gz` — already there from pillar2z run (~2-3 GB)
3. `pillar2y2_epoch_40.pt` — already there (~143 MB)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_path_b.tar.gz /content/
!cd /content && tar xzf colorlines_path_b.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar2y2_epoch_40.pt',
            '/content/alphatrain/data/model.pt')
print(f'Warm-start: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

t0 = time.time()
!gzip -dc {DRIVE}/v12_pillar2z.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'V12 tensor: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

print(f'Oracle tensor: {os.path.getsize("/content/alphatrain/data/phase1_oracle_path_b.pt")/1e6:.1f} MB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

%cd /content
!python -m pytest alphatrain/tests/test_train_path_b.py -v --tb=short 2>&1 | tail -20

## Run C — λ=0.05 (conservative oracle)

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --oracle-tensor alphatrain/data/phase1_oracle_path_b.pt \
    --oracle-lambda 0.05 --oracle-beta 10.0 \
    --oracle-noise-floor 0.05 --oracle-scale 0.20 \
    --oracle-batch-size 4096 \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --color-augment \
    --epochs 10 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --copy-to /content/drive/MyDrive/alphatrain/path_b_C_smoke_best.pt \
    --save-dir /content/checkpoints/path_b_C_smoke

In [ ]:
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'path_b_C_smoke'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')

## Run D — λ=0.10 (aggressive oracle)

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --oracle-tensor alphatrain/data/phase1_oracle_path_b.pt \
    --oracle-lambda 0.10 --oracle-beta 10.0 \
    --oracle-noise-floor 0.05 --oracle-scale 0.20 \
    --oracle-batch-size 4096 \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --color-augment \
    --epochs 10 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --copy-to /content/drive/MyDrive/alphatrain/path_b_D_smoke_best.pt \
    --save-dir /content/checkpoints/path_b_D_smoke

In [ ]:
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'path_b_D_smoke'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')